# 🎯 Project: Traffic Sensor Height Calibration Impact Analysis (120cm vs 100cm)

| **Project Metadata** | **Details** |
| :--- | :--- |
| **Date** | December 2025 |
| **Business Unit** | Decathlon Turkey (TR) |
| **Context** | Global Requirement (Lowering sensors from 120cm to 100cm) |
| **Methodology** | A/B test (Control Group) & Difference-in-Differences (DiD) |


## 1. 🎯 EXECUTIVE SUMMARY
The project of lowering the lower limit of traffic sensors at store entrances from **120cm to 100cm** to include "child visitors" in the traffic data was tested.

**General Result:** The change **did not create a statistically significant traffic increase** across Turkey (P-Value > 0.05).

**Key Findings:** Sensor interaction was observed only in the **Ataşehir** store, where child density is high; however, this situation disrupted KPI tracking by lowering the Conversion Rate (CR) by **1 point**.

**Decision:** 


## 2. 🧪 Methodology & Setup
This study is not a simple "Pre/Post" comparison. A scientific approach, isolated from seasonality and external factors, has been applied.

* **Test Period:** December 8 - 14, 2025 (Sensors were set to 100cm)

* **Test Group:** 5 stores with varying child densities (Ataşehir, Korupark, Espark, Levent, Ankamall)

* **Control Group:** 5 stores matched with Test stores based on Child Product Sales Ratio (Child Ratio) and Historical Trend (r > 0.90) (OneTower, Serdivan, Gaziemir, Nautilus, Carousel).


## 3. ⚖️ Store Selection & Pairing
Control stores were not selected randomly; they were determined algorithmically based on **Pearson Correlation** and **Child Product Sales Ratio (Child Ratio)**.

| **Test Store (100cm)** | **Control Store (120cm)** | 
| :--- | :--- | 
| **ATASEHIR (976)** | **ONETOWER (2466)** | 0.89 |
| **KORUPARK (1935)** | **MEYDAN (1513)** | 0.99 |
| **ESPARK (2328)** | **MAMAK (1070)** | 0.98 | 
| **LEVENT (1604)** | **ADANA (793)** | 0.97 | 
| **ANKAMALL (1097)** | **BAYRAMPASA (538)** | 

> **Note:** Both geographical region (climate) and Mall type (Family Mall vs. Business Center) were considered in the pairing process to ensure robustness.


### Child Rate


In [0]:
%sql
SELECT
economical_businessunit_name AS Magaza_Adi,
--label_customer_behavior_segment_historical,
COUNT(
CASE
WHEN array_contains(di.gender, 'BABY BOYS')
OR array_contains(di.gender, 'BABY GIRLS')
OR array_contains(di.gender, 'BOYS')
OR array_contains(di.gender, 'GIRLS')
OR array_contains(di.gender, 'KIDS')
THEN sal.transaction_id
END
) AS Cocuk_model_adedi,
COUNT(sal.transaction_id) AS Total_Transaction_Count, 
CAST(
COUNT(
CASE
WHEN array_contains(di.gender, 'BABY BOYS')
OR array_contains(di.gender, 'BABY GIRLS')
OR array_contains(di.gender, 'BOYS')
OR array_contains(di.gender, 'GIRLS')
OR array_contains(di.gender, 'KIDS')
THEN sal.transaction_id
END
) AS DOUBLE
) / COUNT(sal.transaction_id) AS Cocuk_Total_Orani
FROM
prod_datalake_gold_sales.sales_detail_migration_version sal
INNER JOIN hive_metastore.prod_datalake_silver.f_member_purchase mp ON mp.transaction_id = sal.transaction_id
INNER JOIN hive_metastore.prod_datalake_insight_ml_customer_behavioral_segmentation.f_customer_behavior_segment cbs ON mp.member_id = cbs.member_id
INNER JOIN hive_metastore.datalake_crm_parquet.d_customers_by_country dd ON dd.member_id = cbs.member_id
INNER JOIN hive_metastore.prod_datalake_insight_analytics_core_datamart.dim_item di ON di.model_code = sal.model_code
WHERE
year IN (2025)
AND dd.cnt_country_code_creator = 'TR'
AND cbs.is_behavior_segment_active = true
AND sal.gmv_recorded_at BETWEEN '2025-01-01' AND '2025-11-30'
AND economical_businessunit_country_code = 'TR'
AND merchant_businessunit_id <> '7-2991-2991'
GROUP BY ALL

### All Traffic

In [0]:
import pandas as pd

sql_query = """
SELECT 
    store_id as store_code,
    to_date(start_time) as n_date,
    sum(enters) as traffic
FROM 
    prod_datalake_gold_store_traffic.in_and_out 
WHERE 
    country = 'TR'
    AND to_date(start_time) BETWEEN '2025-09-01' AND '2025-12-07' 
    AND provider='NEDAP'
    AND hour(start_time) BETWEEN 10 and 21
GROUP BY 
    1, 2
ORDER BY 
    1, 2
"""
df = spark.sql(sql_query).toPandas()

# Tarih formatını düzenleme
df['n_date'] = pd.to_datetime(df['n_date'])

# Mağaza kodlarını string'e çevirme
df['store_code'] = df['store_code'].astype(str)

### Correlation Matrix (Traffic Trend & Child Ratio)
### 
_Toplam Skor = Trafik Trendi(r) * %70 + Çocuk Oranı Benzerliği * %30_

In [0]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. RESMİ MAĞAZA KOD HARİTASI (Sizin Listenizden)
# ---------------------------------------------------------
code_map = {
    '538': 'BAYRAMPASA-TURKSPORT',
    '575': 'KENTPARK-TURKSPORT',
    '707': 'GAZIEMIR-TURKSPORT',
    '755': 'SAMSUN-TURKSPORT',
    '793': 'ADANA',
    '813': 'Merter-TURKSPORT',
    '843': 'MERSIN-TURKSPORT',
    '845': 'BASAKSEHIR-TURKSPORT', # Dikkat: Daha önce Meydan sanıyorduk, Başakşehirmiş.
    '864': 'BODRUM-TURKSPORT',
    '976': 'ATASEHIR',
    '1000': 'ANTALYA',
    '1070': 'MAMAK',
    '1095': 'IZMIT',
    '1097': 'ANKAMALL',
    '1331': 'BALCOVA',
    '1363': 'CORLU',
    '1364': 'ISKENDERUN',
    '1386': 'ERASTA',
    '1502': 'MALTEPE', # Dikkat: Daha önce Metromall sanıyorduk.
    '1513': 'MEYDAN',
    '1604': 'LEVENT',
    '1619': 'MARKA',
    '1911': 'VADISTANBUL',
    '1913': 'METROMALL',
    '1923': 'CAROUSEL',
    '1931': 'TORIUM',
    '1932': 'VIAPORT',
    '1935': 'KORUPARK',
    '1936': 'TERRACITY',
    '1942': 'ANTARES',
    '1943': 'KAYSERI',
    '1980': 'OPTIMUM ISTANBUL',
    '2210': 'ISTIKLAL',
    '2328': 'ESPARK',
    '2438': 'AKMERKEZ',
    '2443': 'KARSIYAKA',
    '2465': 'NAUTILUS',
    '2466': 'ONETOWER',
    '2468': 'Vialand',
    '2713': 'SANKOPARK GAZIANTEP',
    '2714': 'KONYA KENT PLAZA', 
    '2868': 'PAZARYERI',
    '3053': 'Diyarbakir',
    '3052': 'Point Bornova',
    '3051': 'CEVAHIR',
    '3118': 'CITYS',
    '3182': 'ONBURDA BALIKESIR',
    '3183': 'TRABZON',
    '3259': 'ANTALYA BEACH PARK DECATHLON',
    '3229': 'SERDIVAN',
    '3359': 'MARKANTALYA',
    '3444': 'ATLANTIS',
    '3442': 'DENIZLI'
}

# ---------------------------------------------------------
# 2. ÇOCUK ORANLARI (İsimleri Code Map ile Eşleştirdim)
# ---------------------------------------------------------
child_ratios = {
    'ATASEHIR': 0.2131, 'IZMIT': 0.2141, 'ONETOWER': 0.2123, 
    'AKMERKEZ': 0.2110, 'METROMALL': 0.2065, 'VADISTANBUL': 0.2041,
    'MEYDAN': 0.2015, 'KORUPARK': 0.2015, 'BASAKSEHIR-TURKSPORT': 0.2007,
    'Vialand': 0.1991, 'KAYSERI': 0.1960, 'ATLANTIS': 0.1948,
    'ANTARES': 0.1922, 'SERDIVAN': 0.1914, 'VIAPORT': 0.1913,
    'KENTPARK-TURKSPORT': 0.1907, 'CORLU': 0.1897, 'MARKA': 0.1849,
    'TORIUM': 0.1849, 'SAMSUN-TURKSPORT': 0.1825, 'MALTEPE': 0.1819,
    'KONYA KENT PLAZA': 0.1815, 'KARSIYAKA': 0.1781, 'ONBURDA BALIKESIR': 0.1777,
    'MAMAK': 0.1777, 'ESPARK': 0.1744, 'Point Bornova': 0.1742,
    'DENIZLI': 0.1740, 'TERRACITY': 0.1735, 'Merter-TURKSPORT': 0.1728,
    'TRABZON': 0.1720, 'OPTIMUM ISTANBUL': 0.1717, 'Diyarbakir': 0.1679,
    'MERSIN-TURKSPORT': 0.1657, 'BALCOVA': 0.1630, 'ERASTA': 0.1622,
    'GAZIEMIR-TURKSPORT': 0.1612, 'ANTALYA': 0.1612, 'CITYS': 0.1572,
    'SANKOPARK GAZIANTEP': 0.1562, 'BAYRAMPASA-TURKSPORT': 0.1537,
    'BODRUM-TURKSPORT': 0.1514, 'ANKAMALL': 0.1487, 'NAUTILUS': 0.1470,
    'LEVENT': 0.1462, 'CAROUSEL': 0.1457, 'ADANA': 0.1411,
    'MARKANTALYA': 0.1322, 'ANTALYA BEACH PARK DECATHLON': 0.1259,
    'CEVAHIR': 0.1176, 'ISTIKLAL': 0.0799
}

# Test Mağazaları (Kodları)
test_stores_codes = ['976', '1935', '2328', '1604', '1097']

# ---------------------------------------------------------
# 3. HESAPLAMA MOTORU (SQL Verisi ile Birleştirme)
# ---------------------------------------------------------
# df'in SQL'den geldiğini ve store_code'un string olduğunu varsayıyoruz
df_pivot = df.pivot_table(index='n_date', columns='store_code', values='traffic')
df_pivot = df_pivot.fillna(method='ffill').fillna(0)

print(f"{'TEST MAĞAZASI':<15} | {'ADAY KODU':<10} | {'ADAY İSMİ':<22} | {'TREND(r)':<8} | {'ÇOCUK':<8} | {'SKOR':<6}")
print("-" * 100)

for t_code in test_stores_codes:
    t_name = code_map.get(t_code, "Bilinmiyor")
    t_child_rate = child_ratios.get(t_name, 0)
    
    # Test mağazasının verisi yoksa atla
    if t_code not in df_pivot.columns: 
        print(f"Veri Yok: {t_name}")
        continue
        
    target_series = df_pivot[t_code]
    
    matches = []
    
    # Tüm adayları tara (Test mağazaları hariç)
    candidates = [c for c in df_pivot.columns if c not in test_stores_codes]
    
    for c_code in candidates:
        cand_series = df_pivot[c_code]
        
        # 1. TRAFİK KORELASYONU
        corr = target_series.corr(cand_series)
        if corr < 0.85: continue # Düşük korelasyonu baştan ele
        
        # 2. DEMOGRAFİK BENZERLİK
        c_name = code_map.get(c_code, c_code) # İsmi bulamazsa kodu yazar
        c_child_rate = child_ratios.get(c_name)
        
        demog_score = 0
        ratio_disp = "N/A"
        
        if c_child_rate is not None and t_child_rate > 0:
            diff = abs(t_child_rate - c_child_rate)
            # %0 fark = 1 puan, %10 fark = 0 puan
            demog_score = max(0, 1 - (diff * 10))
            ratio_disp = f"%{c_child_rate*100:.1f}"
            
            # HİBRİT SKOR: %70 Trend + %30 Demografi
            final_score = (corr * 0.70) + (demog_score * 0.30)
        else:
            # Demografi verisi yoksa skoru %10 cezalandır (Belirsizlik Payı)
            final_score = corr * 0.90
            
        matches.append({
            'Code': c_code, 'Name': c_name, 'Corr': corr,
            'Ratio': ratio_disp, 'Score': final_score
        })
    
    # En yüksek skora göre sırala ve ilk 3'ü yazdır
    top3 = sorted(matches, key=lambda x: x['Score'], reverse=True)[:3]
    
    for m in top3:
        print(f"{t_name:<15} | {m['Code']:<10} | {m['Name']:<22} | {m['Corr']:.4f}   | {m['Ratio']:<8} | {m['Score']:.4f}")
    print("-" * 100)

## 4. 📊 Analysis


### Visual Proof

In [0]:
import pandas as pd

sql_query = """
SELECT 
    store_id as store_code,
    to_date(start_time) as n_date,
    sum(enters) as traffic
FROM 
    prod_datalake_gold_store_traffic.in_and_out 
WHERE 
    country = 'TR'
    AND to_date(start_time) BETWEEN '2025-09-01' AND '2025-12-17' 
    AND provider='NEDAP'
    AND hour(start_time) BETWEEN 10 and 21
GROUP BY 
    1, 2
ORDER BY 
    1, 2
"""
df = spark.sql(sql_query).toPandas()

# Tarih formatını düzenleme
df['n_date'] = pd.to_datetime(df['n_date'])

# Mağaza kodlarını string'e çevirme
df['store_code'] = df['store_code'].astype(str)

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ---------------------------------------------------------
# 1. VERİYİ HAZIRLAMA (SQL SONUCU)
# ---------------------------------------------------------
# SQL sonucunu df olarak alıyoruz (Burada SQL'i çalıştırdığını varsayıyorum)
df = spark.sql(sql_query).toPandas() 
df['n_date'] = pd.to_datetime(df['n_date'])
df['store_code'] = df['store_code'].astype(str)

# Pivot (Kolay çizim için)
df_pivot = df.pivot_table(index='n_date', columns='store_code', values='traffic').fillna(0)

# ---------------------------------------------------------
# 2. FİNAL "DREAM TEAM" EŞLEŞMELERİ (Resmi Kodlarla)
# ---------------------------------------------------------
final_pairs = {
    'ATASEHIR': {'test': '976',  'control': '2466', 'control_name': 'ONETOWER'},
    'KORUPARK': {'test': '1935', 'control': '1513',  'control_name': 'MEYDAN'},
    'ESPARK':   {'test': '2328', 'control': '1070', 'control_name': 'MAMAK'},
    'LEVENT':   {'test': '1604', 'control': '793', 'control_name': 'ADANA'},
    'ANKAMALL': {'test': '1097', 'control': '538',  'control_name': 'BAYRAMPASA'}
}

# ---------------------------------------------------------
# 3. GRAFİK AYARLARI (Test Dönemi)
# ---------------------------------------------------------
TEST_START = pd.to_datetime('2025-12-08')
TEST_END   = pd.to_datetime('2025-12-14')

# Çizim Başlangıcı (Grafik çok sıkışık olmasın diye Kasım başından itibaren bakalım)
PLOT_START = '2025-11-01' 

# ---------------------------------------------------------
# 4. GRAFİK MOTORU
# ---------------------------------------------------------
plt.figure(figsize=(20, 18)) # Büyük Tuval

for i, (store_name, codes) in enumerate(final_pairs.items()):
    ax = plt.subplot(3, 2, i+1) # 3 Satır, 2 Sütun
    
    test_code = codes['test']
    ctrl_code = codes['control']
    ctrl_name = codes['control_name']
    
    # Veri Kontrolü
    if test_code not in df_pivot.columns or ctrl_code not in df_pivot.columns:
        print(f"⚠️ Veri eksik: {store_name} veya {ctrl_name}")
        continue

    # Veriyi filtrele (Sadece Kasım ve Aralık)
    data = df_pivot.loc[PLOT_START:]
    
    # --- ÇİZİM ---
    # 1. Test Mağazası (Kalın Mavi)
    ax.plot(data.index, data[test_code], 
            label=f'TEST: {store_name}', color='#1f77b4', linewidth=3)
    
    # 2. Kontrol Mağazası (Kesikli Turuncu)
    # Kontrol mağazasını test mağazasıyla aynı seviyeye getirmek için (Normalize)
    # Test başlangıcından önceki ortalamalara göre bir katsayı (scaling) uygulayabiliriz
    # ama ham veriyi görmek daha dürüsttür. O yüzden direkt çiziyoruz.
    ax.plot(data.index, data[ctrl_code], 
            label=f'KONTROL: {ctrl_name}', color='#ff7f0e', linewidth=2.5, linestyle='--')
    
    # --- VURGULAMA ---
    # Test Haftası (Sarı Gölge Alan)
    ax.axvspan(TEST_START, TEST_END, color='yellow', alpha=0.3, label='Test Haftası (8-14 Ara)')
    
    # Sensör Değişim Anı (Kırmızı Çizgi)
    ax.axvline(x=TEST_START, color='red', linestyle=':', linewidth=2)

    # Görsel Süslemeler
    ax.set_title(f"{store_name} vs {ctrl_name}", fontsize=14, fontweight='bold')
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, which='both', linestyle='--', alpha=0.5)
    
    # Tarih Formatı
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d-%b'))
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))

plt.tight_layout()
plt.show()

## 5. 📊 Key Hypotheses
* **H1 (Traffic):** The sensor change will yield a statistically significant increase in traffic (+% Impact).
* **H2 (Conversion):** Since the number of transactions remains constant while traffic increases (denominator effect), the CR (Conversion Rate) will mathematically decrease.
* **H3 (Timing):** The traffic increase will be concentrated particularly on **weekends and after-school hours** (reflecting child mobility)